In [1]:
import os

import chromadb
import dotenv
from agents import Agent, Runner, WebSearchTool, function_tool, trace
from agents.mcp import MCPServerStreamableHttp

dotenv.load_dotenv()

# Setup Exa Search MCP Server
exa_search_mcp = MCPServerStreamableHttp(
    name="Exa Search MCP",
    params={
        "url": f"https://mcp.exa.ai/mcp?exaApiKey={os.environ.get('EXA_API_KEY')}",
        "timeout": 30,
    },
    client_session_timeout_seconds=30,
    cache_tools_list=True,
    max_retry_attempts=1,
)

await exa_search_mcp.connect()

In [2]:
# Connect to existing collection
chroma_client = chromadb.PersistentClient(path="./chroma")
emission_db = chroma_client.get_collection(name="bitcoin_mining_emissions")

In [3]:
@function_tool
def emission_lookup_tool(query: str, max_results: int = 5) -> str:
    """
    Tool function for the internal RAG database to look up emission data collection 
    activities for Bitcoin mining operations (Scope 1, 2, and 3).
    
    Use this tool for:
    - Standard emission tracking activities
    - Data collection requirements
    - Units of measurement
    - Typical percentage impacts
    
    Use Exa Search instead for:
    - Current emission factors
    - Latest regulations
    - Real-world case studies
    - Technology updates

    Args:
        query: The emission activity or category to look up (e.g., "electricity", 
               "ASIC miners", "cooling systems", "Scope 2").
        max_results: The maximum number of results to return (default: 5).

    Returns:
        A string containing the emission tracking information from the database.
    """
    results = emission_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No emission tracking information found for: {query}"

    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        
        emission_source = metadata["emission_source"].title()
        scope = metadata["emission_scope"].upper()
        category = metadata["activity_category"].replace("_", " ").title()
        data_to_collect = metadata["data_to_collect"]
        unit = metadata["unit_of_measure"]
        percentage = metadata["percentage_range"]
        priority = metadata["priority_level"].upper()

        formatted_results.append(
            f"""
{i+1}. {emission_source}
   - Scope: {scope}
   - Category: {category}
   - Data to collect: {data_to_collect}
   - Unit: {unit}
   - Typical % of emissions: {percentage}%
   - Priority: {priority}
            """.strip()
        )

    return "Emission Tracking Activities (from internal database):\n\n" + "\n\n".join(formatted_results)


In [4]:
@function_tool
def emission_filter_tool(
    scope: str = None, 
    priority_level: str = None, 
    min_percentage: float = None,
    max_results: int = 10
) -> str:
    """
    Tool function to filter emission activities by specific criteria from the internal database.
    
    Use this for precise filtering when you need activities matching specific criteria.

    Args:
        scope: Filter by emission scope - options: "scope1", "scope2", "scope3"
        priority_level: Filter by priority - options: "critical", "high", "medium", "low-medium", "low"
        min_percentage: Filter by minimum percentage impact (e.g., 1.0 for activities >1%)
        max_results: The maximum number of results to return (default: 10)

    Returns:
        A string containing filtered emission tracking information.
    """
    where_clause = {}
    
    if scope:
        where_clause["emission_scope"] = scope.lower()
    
    if priority_level:
        where_clause["priority_level"] = priority_level.lower()
    
    if min_percentage is not None:
        where_clause["percentage_min"] = {"$gte": min_percentage}
    
    query_parts = []
    if scope:
        query_parts.append(f"{scope} emissions")
    if priority_level:
        query_parts.append(f"{priority_level} priority")
    if min_percentage:
        query_parts.append(f"high impact activities")
    
    query_text = " ".join(query_parts) if query_parts else "emission activities"
    
    results = emission_db.query(
        query_texts=[query_text], 
        n_results=max_results,
        where=where_clause if where_clause else None
    )

    if not results["documents"][0]:
        return f"No emission activities found matching the criteria."

    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        
        emission_source = metadata["emission_source"].title()
        scope_label = metadata["emission_scope"].upper()
        percentage = metadata["percentage_range"]
        priority = metadata["priority_level"].upper()
        unit = metadata["unit_of_measure"]

        formatted_results.append(
            f"{i+1}. {emission_source} ({scope_label}) - {percentage}% impact - Priority: {priority} - Unit: {unit}"
        )

    return "Filtered Emission Activities (from internal database):\n\n" + "\n".join(formatted_results)

1st Agent: Emission Data Collection Advisor

In [5]:
emission_data_advisor_agent = Agent(
    name="Emission Data Collection Advisor",
    instructions="""
    You are a specialized assistant helping with emission data collection for Bitcoin mining operations.
    You provide guidance on what data to collect according to GHG Protocol standards.
    
    **Your workflow:**
    0) First, use emission_lookup_tool to check the internal database for the emission tracking 
       activities requested in the query. Use the result if it matches the user's query.
    1) If you need more context about current industry practices, emission factors, or specific 
       technology details, use Exa Search to find:
       - Current emission factors for electricity grids
       - Latest equipment specifications (ASIC miners, cooling systems)
       - Industry standards and best practices
       - Regulatory requirements
    2) After web search, use emission_lookup_tool or emission_filter_tool to cross-reference 
       with GHG Protocol standards.
    3) For each emission source mentioned, provide:
       - What data to collect
       - Unit of measurement
       - Collection frequency (daily, monthly, annually)
       - Priority level
       - Typical percentage impact
    
    **Key guidelines:**
    - Always specify Scope 1 vs Scope 2 vs Scope 3
    - Include units of measurement
    - Mention priority levels (CRITICAL, HIGH, MEDIUM, LOW)
    - Be specific about data collection methods
    - Don't use emission tools more than 10 times per query
    
    Give concise, actionable answers.
    """,
    tools=[emission_lookup_tool, emission_filter_tool],
    mcp_servers=[exa_search_mcp],
)

2nd Agent: Emission Reduction Strategy Planner

In [6]:
emission_reduction_planner_agent = Agent(
    name="Emission Reduction Strategy Planner",
    instructions="""
    You are a helpful assistant that develops emission reduction strategies for Bitcoin mining operations.
    You give concise, strategic answers.
    
    Given the user's mining operation details (capacity, location, current setup), develop 
    practical emission reduction strategies that:
    - Reduce Scope 1 emissions (on-site combustion, fugitive emissions)
    - Reduce Scope 2 emissions (electricity consumption)
    - Are technically feasible for Bitcoin mining
    - Consider operational continuity (24/7 uptime requirements)
    
    **Strategy categories to consider:**
    1. Renewable energy integration (solar, wind, hydro)
    2. Grid selection and PPAs (Power Purchase Agreements)
    3. Cooling efficiency improvements (PUE optimization)
    4. ASIC hardware efficiency upgrades
    5. Waste heat recovery and reuse
    6. Demand response and load shifting
    
    **For each strategy, explicitly mention:**
    - Strategy name
    - Primary scope impacted (Scope 1, 2, or 3)
    - Expected emission reduction percentage
    - Why this is an effective choice for Bitcoin mining
    - Implementation complexity (Low, Medium, High)
    
    Be realistic about costs, timelines, and operational impacts.
    """,
)

Convert agents to tools:

In [7]:
emission_data_collection_tool = emission_data_advisor_agent.as_tool(
    tool_name="emission-data-collection-advisor",
    tool_description="Use this tool to determine what emission data to collect for specific mining activities and equipment",
)

emission_reduction_strategy_tool = emission_reduction_planner_agent.as_tool(
    tool_name="emission-reduction-strategy-planner",
    tool_description="Use this tool to develop emission reduction strategies for Bitcoin mining operations",
)

3rd Agent: Emission Factor & Cost Estimator

In [8]:
emission_cost_estimator_agent = Agent(
    name="Emission Cost Estimator",
    instructions="""
    You are a helpful assistant that estimates carbon costs and emission factors for Bitcoin mining operations.
    
    **Your tasks:**
    1. Take the emission reduction strategies (with expected reductions)
    2. Use web search to find:
       - Current carbon credit prices ($/tCO2e)
       - Grid emission factors for the specified location (gCO2/kWh)
       - Cost of renewable energy PPAs in the region
       - Capital costs for equipment upgrades (cooling, ASICs)
    3. Calculate:
       - Current annual carbon footprint (tCO2e/year)
       - Potential emission reductions (tCO2e/year) for each strategy
       - Carbon cost savings ($/year)
       - ROI timeline for capital investments
    
    **In your final output provide:**
    - Strategy name
    - Current emissions baseline
    - Emission reduction potential (tCO2e/year and %)
    - Carbon cost savings ($/year)
    - Implementation cost estimate
    - Simple payback period
    
    Use markdown tables for clarity. Be as concise and data-driven as possible.
    Cite sources for emission factors and carbon prices.
    """,
    tools=[WebSearchTool()],
)

4th Agent: Master Emissions Optimization Coordinator

In [9]:
emissions_optimization_coordinator = Agent(
    name="Emissions Optimization Coordinator",
    instructions="""
    You are the master coordinator for Bitcoin mining emissions optimization.
    You help mining operators develop comprehensive carbon reduction plans with:
    1. Complete emission data collection requirements
    2. Practical reduction strategies
    3. Financial analysis and ROI projections
    
    **Follow this workflow carefully:**
    
    STEP 1: Understand the operation
    - Mining capacity (MW or TH/s)
    - Location (for grid emission factors)
    - Current power sources (grid, on-site generation, renewables)
    - Current PUE if available
    
    STEP 2: Use emission_data_collection_tool to determine:
    - All critical Scope 1 and Scope 2 data collection activities
    - What metrics to track (kWh, kg refrigerant, liters fuel, etc.)
    - Priority levels for each activity
    
    STEP 3: Use emission_reduction_strategy_tool to develop:
    - 3-5 practical emission reduction strategies
    - Strategies should cover both Scope 1 and Scope 2
    - Mix of quick wins and longer-term investments
    
    STEP 4: Handoff to Emission Cost Estimator to:
    - Calculate emission reductions (tCO2e/year)
    - Estimate carbon cost savings
    - Provide ROI analysis for each strategy
    
    **Final output format:**
    
    # Emissions Optimization Plan for [Operation Name]
    
    ## 1. Required Data Collection
    [List from emission_data_collection_tool]
    
    ## 2. Recommended Strategies
    [List from emission_reduction_strategy_tool]
    
    ## 3. Financial Analysis
    [Handed off to emission_cost_estimator_agent]
    
    Use clear markdown formatting. Be comprehensive but concise.
    Prioritize strategies by ROI and emission reduction potential.
    """,
    tools=[emission_data_collection_tool, emission_reduction_strategy_tool],
    handoff_description="""
    Provide the emission reduction strategies with expected reduction percentages,
    the mining operation details (capacity, location), and current power sources
    for financial analysis and carbon cost estimation.
    """,
    handoffs=[emission_cost_estimator_agent],
)

In [10]:
# Example 1: Small mining operation
with trace("Multi Agent: Small Mining Operation"):
    result = await Runner.run(
        emissions_optimization_coordinator,
        """
        I'm operating a 5 MW Bitcoin mining facility in Jakarta, Indonesia.
        Currently using 100% grid electricity with a PUE of 1.3.
        I want to reduce my carbon footprint. What data should I collect,
        what strategies should I implement, and what's the financial impact?
        Give me three practical options.
        """
    )
    print(result.final_output)

I can produce a tight, data-driven plan, but I need to confirm a couple of key inputs to compute precise numbers:

- Baseline electricity use: Do you want to use your stated 5 MW load (IT miner) with PUE 1.3 as the baseline, giving total facility power of 6.5 MW? That yields 56,940 MWh/year.
- Location: You’re in Jakarta, Indonesia. Do you want location-based (grid average) CO2e factors only, or include any market-based factors (if you have PPAs/RECs)?
- Available data for costs: Do you want rough market benchmarks for:
  - ASIC upgrades (new-gen miners)
  - Cooling hardware upgrades
  - On-site solar PV (rooftop/land) and storage
  - PPAs with grid/sellers in Indonesia
  - Backup genset fuel-switch options

If you approve, I will:
- Use web sources to fetch: current carbon credit prices ($/tCO2e), Jakarta/Indonesia grid emission factors (gCO2/kWh), regional renewable energy PPAs costs, and capital costs for cooling upgrades and newer ASICs.
- Compute:
  - Current annual carbon footpri

In [ ]:
# Example 2: Large operation with mixed power
with trace("Multi Agent: Large Mixed Power Operation"):
    result = await Runner.run(
        emissions_optimization_coordinator,
        """
        We have a 50 MW Bitcoin mining facility in Texas with:
        - 70% grid electricity
        - 30% natural gas generators (Scope 1)
        - PUE of 1.15 (immersion cooling)
        
        What emission data should we track, and what are the top 3 strategies
        to reduce our carbon footprint? Include cost analysis.
        """
    )
    print(result.final_output)

In [ ]:
# Example 3: Greenfield project
with trace("Multi Agent: New Facility Planning"):
    result = await Runner.run(
        emissions_optimization_coordinator,
        """
        I'm planning a new 20 MW facility in Iceland where geothermal and
        hydro power are available. What emission tracking should I implement
        from day one, and what design choices will minimize my carbon footprint?
        Include financial projections.
        """
    )
    print(result.final_output)

In [ ]:
# Example 4: Flare gas mining
with trace("Multi Agent: Flare Gas Operation"):
    result = await Runner.run(
        emissions_optimization_coordinator,
        """
        We're considering using flare gas from oil fields to power 10 MW of
        Bitcoin mining in North Dakota. What emissions should we track for
        this setup (considering the counterfactual of gas being flared anyway),
        and how does this compare to grid electricity? What's the carbon impact?
        """
    )
    print(result.final_output)

In [ ]:
# Example 5: Cooling optimization focus
with trace("Multi Agent: Cooling Optimization"):
    result = await Runner.run(
        emissions_optimization_coordinator,
        """
        Our 15 MW facility in Singapore has high cooling costs and PUE of 1.4
        due to tropical climate. What emission data should we track for our
        cooling systems, and what strategies can reduce both our PUE and
        carbon footprint? Include ROI analysis.
        """
    )
    print(result.final_output)